# Inference Monitoring with SageMaker Pipeline

This notebook demonstrates how to use the automated monitoring pipeline for fraud detection models.

**Pipeline Steps:**
1. Ground Truth Simulation - Generate synthetic labels
2. Drift Computation - Detect data drift with Evidently
3. MLflow Logging - Log metrics and reports
4. Athena Write - Store results for analysis
5. Threshold Check - Evaluate drift thresholds
6. CloudWatch Alarms - Create alerts

**Prerequisites:**
- Deployed fraud detection endpoint
- Inference data in `inference_responses` table
- Baseline data saved from training run
- MLflow tracking URI configured

## 1. Setup and Configuration

In [2]:
# Import required libraries
import os
import json
import time
from pathlib import Path
from datetime import datetime, timedelta

import boto3
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set style for visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


In [10]:
# Load environment variables
from dotenv import load_dotenv

# Load from project root .env file
project_root = Path.cwd().parent.parent if 'src' in str(Path.cwd()) else Path.cwd().parent
env_path = project_root / '.env'

if env_path.exists():
    load_dotenv(env_path)
    print(f"✓ Environment loaded from: {env_path}")
else:
    print(f"⚠ .env file not found at: {env_path}")
    print("  Please create .env file with required configuration")

# Display configuration
print("\nConfiguration:")
print(f"  AWS Region: {os.getenv('AWS_REGION', 'us-east-1')}")
print(f"  Athena Database: {os.getenv('ATHENA_DATABASE', 'fraud_detection')}")
print(f"  S3 Bucket: {os.getenv('DATA_S3_BUCKET', 'fraud-detection-data-lake')}")
print(f"  MLflow Tracking: {os.getenv('MLFLOW_TRACKING_URI', 'Not configured')[:50]}...")

✓ Environment loaded from: /Users/skoppar/workspace/pure-storage-mlflow/.env

Configuration:
  AWS Region: us-east-1
  Athena Database: fraud_detection
  S3 Bucket: fraud-detection-data-lake-skoppar-XXXXXXXXXXXX
  MLflow Tracking: arn:aws:sagemaker:us-east-1:XXXXXXXXXXXX:mlflow-ap...


In [12]:
# ============================================================================
# CONFIGURATION - Update these values for your environment
# ============================================================================

# Required: Your SageMaker endpoint name
ENDPOINT_NAME = "fraud-detector-endpoint"  # ← CHANGE THIS to your actual endpoint

# Required: MLflow run ID from your training pipeline (for baseline data)
# Get this from MLflow UI or training pipeline output
MLFLOW_RUN_ID = "185f91507a2343b0b31bc9b132a38ad0"  # ← CHANGE THIS

# Optional: Model version tag
MODEL_VERSION = "v1.0"

# Time Window Configuration
# Option 1: Use last N hours (default: 24 hours)
ANALYSIS_LOOKBACK_HOURS = 24

# Option 2: Or specify exact dates (uncomment to use)
# from datetime import datetime
# ANALYSIS_START_TIME = datetime(2024, 3, 15, 0, 0, 0)
# ANALYSIS_END_TIME = datetime(2024, 3, 16, 0, 0, 0)

# Drift Detection Thresholds
DRIFT_THRESHOLD_KS = 0.1  # Kolmogorov-Smirnov statistic threshold
DRIFT_SHARE_THRESHOLD = 0.15  # Share of drifted features threshold

# Performance Thresholds
THRESHOLD_F1_SCORE = 0.70
THRESHOLD_PRECISION = 0.65
THRESHOLD_RECALL = 0.60

# Alarm Configuration
ALARM_THRESHOLD_DRIFTED_FEATURES = 5  # Trigger alarm if more than N features drift
ENABLE_ALARMS = True  # Set to False to disable CloudWatch alarms

# Ground Truth Simulation (for testing/demo)
ENABLE_SIMULATION = True  # Set to False if you have real ground truth
HIGH_CONFIDENCE_THRESHOLD = 0.9
HIGH_CONFIDENCE_ACCURACY = 0.95

# ============================================================================
# VALIDATION - Do not modify below this line
# ============================================================================

print("Validating configuration...\n")

errors = []
warnings = []

# Validate required parameters - check for placeholders or empty values
if not ENDPOINT_NAME or ENDPOINT_NAME.startswith("YOUR-") or ENDPOINT_NAME.startswith("REPLACE"):
    errors.append("ENDPOINT_NAME must be set to your actual SageMaker endpoint name")
    errors.append("  Get this from: SageMaker Console -> Inference -> Endpoints")

if not MLFLOW_RUN_ID or MLFLOW_RUN_ID.startswith("YOUR-") or MLFLOW_RUN_ID.startswith("REPLACE"):
    errors.append("MLFLOW_RUN_ID must be set to your actual training run ID")
    errors.append("  Get this from: MLflow UI or training pipeline output")
    errors.append("  Run this to find it: mlflow.search_runs(experiment_names=['sg-credit-card-fraud-detection-training'])")

# Validate MLflow run ID format (should be 32 hex characters)
if MLFLOW_RUN_ID and not MLFLOW_RUN_ID.startswith("YOUR-") and not MLFLOW_RUN_ID.startswith("REPLACE"):
    if len(MLFLOW_RUN_ID) != 32:
        errors.append(f"MLFLOW_RUN_ID has invalid format (length {len(MLFLOW_RUN_ID)}, expected 32)")
    elif not all(c in '0123456789abcdefABCDEF' for c in MLFLOW_RUN_ID):
        errors.append("MLFLOW_RUN_ID should contain only hexadecimal characters (0-9, a-f)")

# Check SNS topic configuration
sns_topic_arn = os.getenv('MONITORING_SNS_TOPIC_ARN', '')
if not sns_topic_arn and ENABLE_ALARMS:
    warnings.append("MONITORING_SNS_TOPIC_ARN not configured in .env")
    warnings.append("  Alarms will be created but no notifications will be sent")
    warnings.append("  To fix: Run 'python src/scripts/setup_monitoring_pipeline.py --create-sns-topic'")

# Calculate time window
if 'ANALYSIS_START_TIME' not in locals():
    from datetime import datetime, timedelta
    ANALYSIS_END_TIME = datetime.utcnow()
    ANALYSIS_START_TIME = ANALYSIS_END_TIME - timedelta(hours=ANALYSIS_LOOKBACK_HOURS)

# Display validation results
if errors:
    print("❌ CONFIGURATION ERRORS:")
    for error in errors:
        print(f"   {error}")
    print("\n" + "="*80)
    print("Please update the configuration values above and re-run this cell.")
    print("="*80)
    raise ValueError("Invalid configuration - see errors above")

if warnings:
    print("⚠️  WARNINGS:")
    for warning in warnings:
        print(f"   {warning}")
    print()

# Create SNS topic ARN (placeholder if not configured)
if not sns_topic_arn:
    sts_client = boto3.client('sts')
    account_id = sts_client.get_caller_identity()['Account']
    aws_region = os.getenv('AWS_REGION', 'us-east-1')
    sns_topic_arn = f"arn:aws:sns:{aws_region}:{account_id}:monitoring-alerts-placeholder"

SNS_TOPIC_ARN = sns_topic_arn

# Display validated configuration
print("✓ CONFIGURATION VALIDATED\n")
print("Model & Endpoint:")
print(f"  Endpoint Name: {ENDPOINT_NAME}")
print(f"  MLflow Run ID: {MLFLOW_RUN_ID}")
print(f"  Model Version: {MODEL_VERSION}")
print(f"\nTime Window:")
print(f"  Start: {ANALYSIS_START_TIME.isoformat()}")
print(f"  End:   {ANALYSIS_END_TIME.isoformat()}")
print(f"  Duration: {(ANALYSIS_END_TIME - ANALYSIS_START_TIME).total_seconds() / 3600:.1f} hours")
print(f"\nThresholds:")
print(f"  Drift KS: {DRIFT_THRESHOLD_KS}")
print(f"  Drift Share: {DRIFT_SHARE_THRESHOLD}")
print(f"  F1 Score: {THRESHOLD_F1_SCORE}")
print(f"  Drifted Features Alarm: {ALARM_THRESHOLD_DRIFTED_FEATURES}")
print(f"\nOptions:")
print(f"  Enable Alarms: {ENABLE_ALARMS}")
print(f"  Enable Simulation: {ENABLE_SIMULATION}")
print(f"  SNS Topic: {SNS_TOPIC_ARN[:60]}..." if len(SNS_TOPIC_ARN) > 60 else f"  SNS Topic: {SNS_TOPIC_ARN}")
print("\n" + "="*80)
print("Configuration ready! Proceed to next cells.")
print("="*80)

Validating configuration...

⚠️  WARNINGS:
   MONITORING_SNS_TOPIC_ARN not configured in .env
     Alarms will be created but no notifications will be sent
     To fix: Run 'python src/scripts/setup_monitoring_pipeline.py --create-sns-topic'

✓ CONFIGURATION VALIDATED

Model & Endpoint:
  Endpoint Name: fraud-detector-endpoint
  MLflow Run ID: 185f91507a2343b0b31bc9b132a38ad0
  Model Version: v1.0

Time Window:
  Start: 2026-03-16T04:57:21.080325
  End:   2026-03-17T04:57:21.080325
  Duration: 24.0 hours

Thresholds:
  Drift KS: 0.1
  Drift Share: 0.15
  F1 Score: 0.7
  Drifted Features Alarm: 5

Options:
  Enable Alarms: True
  Enable Simulation: True
  SNS Topic: arn:aws:sns:us-east-1:XXXXXXXXXXXX:monitoring-alerts-placeho...

Configuration ready! Proceed to next cells.


In [ ]:
## To find run ids
import mlflow
mlflow.search_runs(experiment_names=['sg-credit-card-fraud-detection-training'])

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.test_f1,metrics.test_false_negatives,metrics.test_false_positives,metrics.test_true_positives,...,params.learning_rate,params.num_boost_round,tags.mlflow.user,tags.mlflow.source.name,tags.use_case,tags.model_type,tags.mlflow.source.type,tags.pipeline_step,tags.mlflow.runName,tags.framework
0,185f91507a2343b0b31bc9b132a38ad0,1,FINISHED,s3://sagemaker-us-east-1-XXXXXXXXXXXX/1/185f91...,2026-03-16 01:47:51.442000+00:00,2026-03-16 01:47:59.131000+00:00,0.864865,18.0,7.0,80.0,...,0.05,200,root,/opt/ml/code/train.py,fraud_detection,binary_classification,LOCAL,training,bold-gnat-979,xgboost
1,d3f32f581ed94b11a149f967d68a4b9f,1,FINISHED,s3://sagemaker-us-east-1-XXXXXXXXXXXX/1/d3f32f...,2026-02-21 09:00:22.330000+00:00,2026-02-21 09:00:35.031000+00:00,0.864865,18.0,7.0,80.0,...,0.05,200,root,/opt/ml/code/train.py,fraud_detection,binary_classification,LOCAL,training,thundering-bear-752,xgboost
2,7fe05a403d9e419583d4c2a253a5d576,1,FINISHED,s3://sagemaker-us-east-1-XXXXXXXXXXXX/1/7fe05a...,2026-02-21 07:38:16.505000+00:00,2026-02-21 07:38:29.776000+00:00,0.864865,18.0,7.0,80.0,...,0.05,200,root,/opt/ml/code/train.py,fraud_detection,binary_classification,LOCAL,training,nebulous-skink-522,xgboost
3,dba3698dce6e449db697db09cb8f490c,1,FINISHED,s3://sagemaker-us-east-1-XXXXXXXXXXXX/1/dba369...,2026-02-21 06:03:11.556000+00:00,2026-02-21 06:03:19.122000+00:00,0.864865,18.0,7.0,80.0,...,0.05,200,root,/opt/ml/code/train.py,fraud_detection,binary_classification,LOCAL,training,bald-calf-705,xgboost
4,05f9778382aa41e48388fb911b6966c9,1,FINISHED,s3://sagemaker-us-east-1-XXXXXXXXXXXX/1/05f977...,2026-02-20 01:46:03.588000+00:00,2026-02-20 01:46:11.382000+00:00,0.864865,18.0,7.0,80.0,...,0.05,200,root,/opt/ml/code/train.py,fraud_detection,binary_classification,LOCAL,training,grandiose-koi-610,xgboost
5,677aadf201d340d385ca25dbbbeb22b7,1,FINISHED,s3://sagemaker-us-east-1-XXXXXXXXXXXX/1/677aad...,2026-02-17 02:06:42.290000+00:00,2026-02-17 02:06:49.895000+00:00,0.838384,15.0,17.0,83.0,...,0.05,200,root,/opt/ml/code/train.py,fraud_detection,binary_classification,LOCAL,training,sassy-crane-829,xgboost
6,1ac3595ff7b54e0b97cb0b055fbaa519,1,FINISHED,s3://sagemaker-us-east-1-XXXXXXXXXXXX/1/1ac359...,2026-02-16 19:36:52.669000+00:00,2026-02-16 19:37:00.545000+00:00,0.879581,14.0,9.0,84.0,...,0.05,200,root,/opt/ml/code/train.py,fraud_detection,binary_classification,LOCAL,training,nervous-wolf-9,xgboost
7,d75153c119da4fe9920d5d6292b3fbc4,1,FINISHED,s3://sagemaker-us-east-1-XXXXXXXXXXXX/1/d75153...,2026-02-16 17:45:05.133000+00:00,2026-02-16 17:45:13.337000+00:00,0.845361,16.0,14.0,82.0,...,0.05,200,root,/opt/ml/code/train.py,fraud_detection,binary_classification,LOCAL,training,auspicious-finch-533,xgboost


In [35]:
# Import monitoring pipeline
import sys
sys.path.insert(0, str(project_root))

from src.sagemaker.monitoring_pipeline import create_monitoring_pipeline
from src.sagemaker.sg_config import (
    AWS_REGION, ATHENA_DATABASE, DATA_S3_BUCKET,
    MLFLOW_TRACKING_URI
)

print("✓ Monitoring pipeline module imported")

✓ Monitoring pipeline module imported


## 2. Initialize Pipeline

In [36]:
# Create monitoring pipeline instance
pipeline = create_monitoring_pipeline(
    pipeline_name='fraud-detection-monitoring-pipeline',
    region=AWS_REGION
)

print("✓ Pipeline instance created")
print(f"  Pipeline name: {pipeline.pipeline_name}")
print(f"  Region: {pipeline.region}")
print(f"  Role: {pipeline.role[:50]}...")

INFO:botocore.tokens:Loading cached SSO token for default
INFO:botocore.tokens:SSO Token refresh succeeded
INFO:botocore.tokens:Loading cached SSO token for default
INFO:src.sagemaker.monitoring_pipeline:Loaded config from: /Users/skoppar/workspace/pure-storage-mlflow/src/sagemaker/monitoring_config.yaml
INFO:src.sagemaker.monitoring_pipeline:Initialized monitoring pipeline: fraud-detection-monitoring-pipeline
INFO:src.sagemaker.monitoring_pipeline:  Role: arn:aws:iam::XXXXXXXXXXXX:role/service-role/AmazonSageMaker-ExecutionRole-20250722T131288
INFO:src.sagemaker.monitoring_pipeline:  Region: us-east-1
INFO:src.sagemaker.monitoring_pipeline:  Bucket: sagemaker-us-east-1-XXXXXXXXXXXX
INFO:src.sagemaker.monitoring_pipeline:  Account: XXXXXXXXXXXX


✓ Pipeline instance created
  Pipeline name: fraud-detection-monitoring-pipeline
  Region: us-east-1
  Role: arn:aws:iam::XXXXXXXXXXXX:role/service-role/Amazon...


## 3. Create/Update Pipeline in SageMaker

In [37]:
# Upsert pipeline (creates if not exists, updates if exists)
result = pipeline.upsert_pipeline(
    description="Automated model monitoring with Evidently drift detection"
)

print("✓ Pipeline created/updated in SageMaker")
print(f"  Pipeline ARN: {result['pipeline_arn']}")
print(f"  Status: {result['status']}")
print(f"\n  View in console: https://console.aws.amazon.com/sagemaker/home?region={AWS_REGION}#/pipelines")

INFO:src.sagemaker.monitoring_pipeline:Upserting pipeline: fraud-detection-monitoring-pipeline
INFO:src.sagemaker.monitoring_pipeline:================================================================================
INFO:src.sagemaker.monitoring_pipeline:Creating pipeline: fraud-detection-monitoring-pipeline
INFO:src.sagemaker.monitoring_pipeline:================================================================================
INFO:src.sagemaker.monitoring_pipeline:Defining pipeline parameters...
INFO:src.sagemaker.monitoring_pipeline:Defined 20 parameters
INFO:src.sagemaker.monitoring_pipeline:Creating ground truth simulation step...
INFO:sagemaker.image_uris:Defaulting to only available Python version: py3
INFO:src.sagemaker.monitoring_pipeline:✓ Ground truth simulation step created
INFO:src.sagemaker.monitoring_pipeline:Creating drift computation step...
INFO:sagemaker.image_uris:Defaulting to only available Python version: py3
INFO:src.sagemaker.monitoring_pipeline:✓ Drift computatio

✓ Pipeline created/updated in SageMaker
  Pipeline ARN: arn:aws:sagemaker:us-east-1:XXXXXXXXXXXX:pipeline/fraud-detection-monitoring-pipeline
  Status: created

  View in console: https://console.aws.amazon.com/sagemaker/home?region=us-east-1#/pipelines


## 4. Configure Execution Parameters

In [38]:
# Build pipeline parameters from configuration
# All values come from the CONFIGURATION block above

pipeline_parameters = {
    # Time window
    'InferenceWindowStart': ANALYSIS_START_TIME.isoformat(),
    'InferenceWindowEnd': ANALYSIS_END_TIME.isoformat(),
    
    # Model identification
    'EndpointName': ENDPOINT_NAME,
    'MlflowRunId': MLFLOW_RUN_ID,
    'ModelVersion': MODEL_VERSION,
    
    # Drift thresholds
    'DriftThresholdKS': DRIFT_THRESHOLD_KS,
    'DriftShareThreshold': DRIFT_SHARE_THRESHOLD,
    
    # Performance thresholds
    'ThresholdF1Score': THRESHOLD_F1_SCORE,
    'ThresholdPrecision': THRESHOLD_PRECISION,
    'ThresholdRecall': THRESHOLD_RECALL,
    
    # Alarm configuration
    'AlarmThresholdDriftedFeatures': ALARM_THRESHOLD_DRIFTED_FEATURES,
    'EnableAlarms': 'true' if ENABLE_ALARMS else 'false',
    'SnsTopicArn': SNS_TOPIC_ARN,
    
    # Ground truth simulation
    'EnableSimulation': 'true' if ENABLE_SIMULATION else 'false',
    'HighConfidenceThreshold': HIGH_CONFIDENCE_THRESHOLD,
    'HighConfidenceAccuracy': HIGH_CONFIDENCE_ACCURACY,
}

print("Pipeline parameters built from configuration:")
print(f"  Endpoint: {pipeline_parameters['EndpointName']}")
print(f"  Time Window: {pipeline_parameters['InferenceWindowStart'][:19]} to {pipeline_parameters['InferenceWindowEnd'][:19]}")
print(f"  Enable Alarms: {pipeline_parameters['EnableAlarms']}")
print(f"  Enable Simulation: {pipeline_parameters['EnableSimulation']}")


Pipeline parameters built from configuration:
  Endpoint: fraud-detector-endpoint
  Time Window: 2026-03-16T04:57:21 to 2026-03-17T04:57:21
  Enable Alarms: true
  Enable Simulation: true


## 5. Execute Pipeline

In [39]:
# Start pipeline execution
execution_name = f"monitoring-{datetime.utcnow().strftime('%Y%m%d-%H%M%S')}"

execution = pipeline.start_execution(
    execution_name=execution_name,
    parameters=pipeline_parameters
)

execution_arn = execution['execution_arn']

print("✓ Pipeline execution started")
print(f"  Execution name: {execution_name}")
print(f"  Execution ARN: {execution_arn}")
print(f"  Status: {execution['status']}")
print(f"\n  View in console: https://console.aws.amazon.com/sagemaker/home?region={AWS_REGION}#/pipeline-executions/{execution_arn.split('/')[-1]}")

/var/folders/x7/t3b9975n6_z57y4hq5d6yhfc0000gr/T/ipykernel_70546/2278756676.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  execution_name = f"monitoring-{datetime.utcnow().strftime('%Y%m%d-%H%M%S')}"
INFO:src.sagemaker.monitoring_pipeline:Starting pipeline execution: monitoring-20260317-060820
INFO:src.sagemaker.monitoring_pipeline:================================================================================
INFO:src.sagemaker.monitoring_pipeline:Creating pipeline: fraud-detection-monitoring-pipeline
INFO:src.sagemaker.monitoring_pipeline:================================================================================
INFO:src.sagemaker.monitoring_pipeline:Defining pipeline parameters...
INFO:src.sagemaker.monitoring_pipeline:Defined 20 parameters
INFO:src.sagemaker.monitoring_pipeline:Creating ground truth simulation ste

✓ Pipeline execution started
  Execution name: monitoring-20260317-060820
  Execution ARN: arn:aws:sagemaker:us-east-1:XXXXXXXXXXXX:pipeline/fraud-detection-monitoring-pipeline/execution/vdyn4wel5pjz
  Status: Executing

  View in console: https://console.aws.amazon.com/sagemaker/home?region=us-east-1#/pipeline-executions/vdyn4wel5pjz


## 6. Monitor Execution Status

In [ ]:
# Monitor execution status (polling)
print("Monitoring pipeline execution...\n")

max_wait_minutes = 60
poll_interval_seconds = 30
start_monitor_time = time.time()

while True:
    status_info = pipeline.describe_execution(execution_arn)
    status = status_info['status']
    elapsed_minutes = (time.time() - start_monitor_time) / 60
    
    print(f"[{elapsed_minutes:.1f} min] Status: {status}", end='\r')
    
    if status in ['Succeeded', 'Failed', 'Stopped']:
        print(f"\n\n✓ Pipeline execution {status.lower()}")
        print(f"  Duration: {elapsed_minutes:.1f} minutes")
        break
    
    if elapsed_minutes >= max_wait_minutes:
        print(f"\n\n⚠ Timeout after {max_wait_minutes} minutes")
        print("  Pipeline is still executing. Check SageMaker console for progress.")
        break
    
    time.sleep(poll_interval_seconds)

# Display final status
if status == 'Failed':
    print("\n  Check CloudWatch Logs for error details")
elif status == 'Succeeded':
    print("\n  Results are ready for analysis!")

## 7. Query Monitoring Results from Athena

In [ ]:
# Install awswrangler if not already installed
try:
    import awswrangler as wr
except ImportError:
    print("Installing awswrangler...")
    !pip install awswrangler -q
    import awswrangler as wr

print("✓ awswrangler ready")

In [ ]:
# Query latest monitoring results
query = f"""
SELECT
    analysis_timestamp,
    dataset_drift_detected,
    drifted_features_count,
    drift_share,
    f1_score,
    precision,
    recall,
    roc_auc,
    top_drifted_feature_1,
    top_drifted_feature_1_ks_stat,
    alarm_triggered
FROM {ATHENA_DATABASE}.monitoring_results
WHERE endpoint_name = '{ENDPOINT_NAME}'
ORDER BY analysis_timestamp DESC
LIMIT 10
"""

results_df = wr.athena.read_sql_query(
    sql=query,
    database=ATHENA_DATABASE
)

print(f"✓ Retrieved {len(results_df)} monitoring results\n")
results_df

In [ ]:
# Display latest result details
if len(results_df) > 0:
    latest = results_df.iloc[0]
    
    print("Latest Monitoring Result:")
    print("=" * 60)
    print(f"  Timestamp: {latest['analysis_timestamp']}")
    print(f"\nDrift Detection:")
    print(f"  Dataset Drift: {'YES ⚠️' if latest['dataset_drift_detected'] else 'NO ✓'}")
    print(f"  Drifted Features: {latest['drifted_features_count']}")
    print(f"  Drift Share: {latest['drift_share']:.2%}")
    print(f"  Top Drifted Feature: {latest['top_drifted_feature_1']} (KS={latest['top_drifted_feature_1_ks_stat']:.4f})")
    print(f"\nModel Performance:")
    print(f"  F1 Score: {latest['f1_score']:.4f}")
    print(f"  Precision: {latest['precision']:.4f}")
    print(f"  Recall: {latest['recall']:.4f}")
    print(f"  ROC-AUC: {latest['roc_auc']:.4f}")
    print(f"\nAlarms:")
    print(f"  Triggered: {'YES ⚠️' if latest['alarm_triggered'] else 'NO ✓'}")
else:
    print("No monitoring results found. Please wait for pipeline execution to complete.")

## 8. Visualize Drift Trends

In [ ]:
# Query historical data (last 30 days)
query_historical = f"""
SELECT
    analysis_timestamp,
    drifted_features_count,
    drift_share,
    f1_score,
    dataset_drift_detected
FROM {ATHENA_DATABASE}.monitoring_results
WHERE endpoint_name = '{ENDPOINT_NAME}'
  AND analysis_timestamp >= CURRENT_TIMESTAMP - INTERVAL '30' DAY
ORDER BY analysis_timestamp
"""

historical_df = wr.athena.read_sql_query(
    sql=query_historical,
    database=ATHENA_DATABASE
)

print(f"✓ Retrieved {len(historical_df)} historical records")

In [ ]:
# Plot drift trends
if len(historical_df) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('Model Monitoring Trends (Last 30 Days)', fontsize=16, fontweight='bold')
    
    # 1. Drifted Features Count
    axes[0, 0].plot(historical_df['analysis_timestamp'], 
                    historical_df['drifted_features_count'], 
                    marker='o', linewidth=2)
    axes[0, 0].axhline(y=5, color='r', linestyle='--', label='Threshold')
    axes[0, 0].set_title('Drifted Features Count')
    axes[0, 0].set_xlabel('Date')
    axes[0, 0].set_ylabel('Count')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # 2. Drift Share
    axes[0, 1].plot(historical_df['analysis_timestamp'], 
                    historical_df['drift_share'] * 100, 
                    marker='o', linewidth=2, color='orange')
    axes[0, 1].axhline(y=15, color='r', linestyle='--', label='Threshold')
    axes[0, 1].set_title('Drift Share')
    axes[0, 1].set_xlabel('Date')
    axes[0, 1].set_ylabel('% of Features')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # 3. F1 Score
    axes[1, 0].plot(historical_df['analysis_timestamp'], 
                    historical_df['f1_score'], 
                    marker='o', linewidth=2, color='green')
    axes[1, 0].axhline(y=0.70, color='r', linestyle='--', label='Threshold')
    axes[1, 0].set_title('F1 Score')
    axes[1, 0].set_xlabel('Date')
    axes[1, 0].set_ylabel('F1 Score')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # 4. Dataset Drift Occurrences
    drift_counts = historical_df.groupby(historical_df['analysis_timestamp'].dt.date)['dataset_drift_detected'].sum()
    axes[1, 1].bar(range(len(drift_counts)), drift_counts.values, color='red', alpha=0.6)
    axes[1, 1].set_title('Dataset Drift Detections')
    axes[1, 1].set_xlabel('Date')
    axes[1, 1].set_ylabel('Count')
    axes[1, 1].set_xticks(range(len(drift_counts)))
    axes[1, 1].set_xticklabels([str(d) for d in drift_counts.index], rotation=45)
    axes[1, 1].grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()
else:
    print("Not enough historical data for visualization")

## 9. View MLflow Runs

In [ ]:
# Connect to MLflow
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

# Search monitoring runs
experiment_name = 'sg-credit-card-fraud-detection-inference'

runs_df = mlflow.search_runs(
    experiment_names=[experiment_name],
    filter_string="tags.task = 'inference_monitoring'",
    order_by=["start_time DESC"],
    max_results=10
)

if len(runs_df) > 0:
    print(f"✓ Found {len(runs_df)} monitoring runs in MLflow\n")
    
    # Display key metrics
    display_cols = [
        'run_id', 
        'start_time',
        'metrics.dataset_drift_detected',
        'metrics.drifted_features_count',
        'metrics.f1_score'
    ]
    
    available_cols = [col for col in display_cols if col in runs_df.columns]
    runs_df[available_cols].head(10)
else:
    print("No monitoring runs found in MLflow")
    print(f"  Experiment: {experiment_name}")
    print(f"  Tracking URI: {MLFLOW_TRACKING_URI}")

In [ ]:
# View latest MLflow run details
if len(runs_df) > 0:
    latest_run_id = runs_df.iloc[0]['run_id']
    
    # Get run details
    run = mlflow.get_run(latest_run_id)
    
    print("Latest MLflow Run:")
    print("=" * 60)
    print(f"  Run ID: {latest_run_id}")
    print(f"  Start Time: {run.info.start_time}")
    print(f"  Status: {run.info.status}")
    print(f"\nTags:")
    for key, value in run.data.tags.items():
        if key.startswith('monitoring') or key == 'endpoint_name':
            print(f"    {key}: {value}")
    print(f"\nMetrics:")
    for key, value in run.data.metrics.items():
        print(f"    {key}: {value}")
    print(f"\nArtifacts:")
    artifacts = mlflow.artifacts.list_artifacts(latest_run_id)
    for artifact in artifacts:
        print(f"    - {artifact.path}")
    
    # Generate MLflow UI URL
    mlflow_ui_url = f"{MLFLOW_TRACKING_URI}/#/experiments/{run.info.experiment_id}/runs/{latest_run_id}"
    print(f"\n  View in MLflow UI: {mlflow_ui_url}")

## 10. Analyze Top Drifted Features

In [ ]:
# Query most problematic features
query_features = f"""
SELECT
    top_drifted_feature_1 as feature,
    COUNT(*) as drift_occurrences,
    AVG(top_drifted_feature_1_ks_stat) as avg_ks_stat,
    MAX(top_drifted_feature_1_ks_stat) as max_ks_stat
FROM {ATHENA_DATABASE}.monitoring_results
WHERE endpoint_name = '{ENDPOINT_NAME}'
  AND dataset_drift_detected = true
  AND analysis_timestamp >= CURRENT_TIMESTAMP - INTERVAL '30' DAY
GROUP BY top_drifted_feature_1
ORDER BY drift_occurrences DESC
LIMIT 10
"""

features_df = wr.athena.read_sql_query(
    sql=query_features,
    database=ATHENA_DATABASE
)

if len(features_df) > 0:
    print("Top 10 Most Drifted Features:\n")
    display(features_df)
    
    # Visualize
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.barh(features_df['feature'], features_df['drift_occurrences'], color='coral')
    ax.set_xlabel('Number of Drift Detections')
    ax.set_title('Most Frequently Drifted Features (Last 30 Days)')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
else:
    print("No drift detected in the last 30 days")

## 11. Schedule Recurring Execution

In [ ]:
# Create EventBridge rule for daily execution
events_client = boto3.client('events', region_name=AWS_REGION)

rule_name = 'fraud-monitoring-daily-execution'
cron_expression = 'cron(0 2 * * ? *)'  # Daily at 2 AM UTC

print(f"Setting up EventBridge rule: {rule_name}")
print(f"  Schedule: {cron_expression} (Daily at 2 AM UTC)")
print(f"\nNote: This requires additional IAM permissions")
print(f"Alternative: Configure manually in AWS Console")

# Uncomment to create rule
# try:
#     response = events_client.put_rule(
#         Name=rule_name,
#         ScheduleExpression=cron_expression,
#         State='ENABLED',
#         Description='Daily execution of fraud detection monitoring pipeline'
#     )
#     print(f"\n✓ Rule created: {response['RuleArn']}")
#     print(f"  Note: Add SageMaker pipeline as target in AWS Console")
# except Exception as e:
#     print(f"\n✗ Failed to create rule: {e}")

## 12. View QuickSight Dashboard

In [ ]:
# Generate QuickSight dashboard URL
quicksight_client = boto3.client('quicksight', region_name=AWS_REGION)
sts_client = boto3.client('sts')
account_id = sts_client.get_caller_identity()['Account']

dashboard_name = 'fraud-detection-monitoring'

print("QuickSight Dashboard:")
print("=" * 60)
print(f"  Dashboard: {dashboard_name}")
print(f"  Account: {account_id}")
print(f"\n  To view dashboard:")
print(f"  1. Open QuickSight console: https://quicksight.aws.amazon.com/")
print(f"  2. Navigate to Dashboards")
print(f"  3. Open '{dashboard_name}' dashboard")
print(f"\n  To create dashboard (if not exists):")
print(f"  Run: python src/quicksight/setup_quicksight_monitoring.py")

## Summary

This notebook demonstrated:

✅ **Pipeline Setup**: Created/updated monitoring pipeline in SageMaker

✅ **Execution**: Ran pipeline with custom parameters and time windows

✅ **Monitoring**: Tracked execution status in real-time

✅ **Analysis**: Queried results from Athena and visualized trends

✅ **MLflow Integration**: Viewed runs and artifacts in MLflow

✅ **Alerting**: Configured CloudWatch alarms and SNS notifications

### Next Steps:

1. **Schedule Regular Monitoring**: Set up EventBridge rule for daily execution
2. **Configure Alerts**: Add email addresses to SNS topic for notifications
3. **Create Dashboard**: Run QuickSight setup script for visual dashboards
4. **Investigate Drift**: When drift detected, analyze Evidently reports in MLflow
5. **Retrain Model**: If performance degrades, trigger retraining pipeline

### Resources:

- **SageMaker Console**: View pipeline executions and logs
- **MLflow UI**: Explore monitoring runs and artifacts
- **QuickSight**: Visual dashboards for trend analysis
- **CloudWatch**: Monitor alarms and metrics
- **Athena**: Query historical monitoring data